## Configuracion de esta corrida

In [ ]:
INFERIR_SRC = '#!/usr/bin/env python\n"""Referencia para la app: corre el .tflite de MicaTB (G-bacilli) con el mismo preprocesado,\ndecodificacion y NMS que debe hacer la app. Dependencias: numpy, opencv-python y ai-edge-litert\n(o tflite-runtime / tensorflow). No usa ultralytics.\n\nUso: python inferir_tflite.py --model ecoli_gb_E3s43_fp16.tflite --config app_config.json --image campo.jpg [--definicion estricta|amplia]\nSalida (JSON): conteo y cajas [x1,y1,x2,y2] en pixeles de la imagen original, con clase y confianza.\n"""\nimport argparse, json\nimport cv2\nimport numpy as np\n\n\ndef cargar_interprete(path):\n    try:\n        from ai_edge_litert.interpreter import Interpreter\n    except ImportError:\n        try:\n            from tflite_runtime.interpreter import Interpreter\n        except ImportError:\n            from tensorflow.lite import Interpreter\n    it = Interpreter(model_path=path); it.allocate_tensors()\n    return it\n\n\ndef letterbox(bgr, size, fill=114):\n    """Escala al lado mayor = size y rellena con gris centrado (igual que ultralytics, auto=False)."""\n    h, w = bgr.shape[:2]\n    r = min(size / h, size / w)\n    nw, nh = round(w * r), round(h * r)\n    dw, dh = (size - nw) / 2, (size - nh) / 2\n    if (w, h) != (nw, nh):\n        bgr = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)\n    top, bottom = round(dh - 0.1), round(dh + 0.1)\n    left, right = round(dw - 0.1), round(dw + 0.1)\n    out = cv2.copyMakeBorder(bgr, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(fill, fill, fill))\n    return out, r, left, top\n\n\ndef preprocesar(bgr, size):\n    img, r, left, top = letterbox(bgr, size)\n    x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0   # RGB, 0..1\n    return np.transpose(x, (2, 0, 1))[None], r, left, top               # NCHW\n\n\ndef nms_por_clase(boxes, scores, classes, iou_thr):\n    """NMS greedy independiente por clase (suprime si IoU > umbral), como torchvision.batched_nms."""\n    keep = []\n    for c in np.unique(classes):\n        idx = np.where(classes == c)[0]; idx = idx[np.argsort(-scores[idx])]\n        while len(idx):\n            i = idx[0]; keep.append(i)\n            if len(idx) == 1: break\n            rest = idx[1:]\n            xx1 = np.maximum(boxes[i, 0], boxes[rest, 0]); yy1 = np.maximum(boxes[i, 1], boxes[rest, 1])\n            xx2 = np.minimum(boxes[i, 2], boxes[rest, 2]); yy2 = np.minimum(boxes[i, 3], boxes[rest, 3])\n            inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)\n            a1 = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])\n            a2 = (boxes[rest, 2] - boxes[rest, 0]) * (boxes[rest, 3] - boxes[rest, 1])\n            idx = rest[inter / (a1 + a2 - inter + 1e-9) <= iou_thr]\n    return np.array(keep, dtype=int)\n\n\ndef detectar(it, bgr, cfg, definicion="estricta"):\n    size = cfg["entrada"]["tamano"]; H, W = bgr.shape[:2]\n    x, r, left, top = preprocesar(bgr, size)\n    i0, o0 = it.get_input_details()[0], it.get_output_details()[0]\n    it.set_tensor(i0["index"], x); it.invoke()\n    y = it.get_tensor(o0["index"])[0]                                   # [4 + n_clases, n_anclas]\n    cx, cy, w, h = y[0] * size, y[1] * size, y[2] * size, y[3] * size   # cajas normalizadas 0..1 -> px de entrada\n    boxes = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)\n    scores_all = y[4:].T; cls = scores_all.argmax(1); conf = scores_all.max(1)\n    d = cfg["definiciones"][definicion]\n    m = conf > d["conf"]; boxes, conf, cls = boxes[m], conf[m], cls[m]\n    k = nms_por_clase(boxes, conf, cls, d["iou_nms"])[: cfg["max_det"]]\n    boxes, conf, cls = boxes[k], conf[k], cls[k]\n    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - left) / r; boxes[:, [1, 3]] = (boxes[:, [1, 3]] - top) / r   # deshacer letterbox\n    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, W); boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, H)\n    clases_contadas = d["clases_contadas"]\n    contar = np.isin(cls, clases_contadas)\n    return dict(conteo=int(contar.sum()),\n                detecciones=[dict(clase=cfg["clases"][int(c)], conf=round(float(s), 4), caja=[round(float(v), 1) for v in b])\n                             for b, s, c in zip(boxes, conf, cls)])\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser(); ap.add_argument("--model", required=True); ap.add_argument("--config", required=True)\n    ap.add_argument("--image", required=True); ap.add_argument("--definicion", default="estricta", choices=["estricta", "amplia"])\n    a = ap.parse_args(); cfg = json.load(open(a.config, encoding="utf8"))\n    img = cv2.imread(a.image); assert img is not None, "no se pudo leer la imagen"\n    print(json.dumps(detectar(cargar_interprete(a.model), img, cfg, a.definicion), indent=1, ensure_ascii=False))\n'

## Funciones comunes (eclib)

In [ ]:
import os, json, time, shutil, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd, cv2

CLASS_NAMES = ["G-cocci", "G+cocci", "G-bacilli", "G+bacilli"]
GB, GPB = 2, 3          # ids de G-bacilli y G+bacilli
NMS_IOU = 0.7

def prepare_ds(src_root, dst_root="/kaggle/temp/yolo_ds", with_bg=False):
    """Copia el dataset a un lugar escribible y escribe el yaml con rutas reales."""
    src_root, dst_root = Path(src_root), Path(dst_root)
    if not dst_root.exists(): shutil.copytree(src_root, dst_root)
    names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
    tr = "[images/train, images/bg_train]" if with_bg else "images/train"
    y = dst_root / ("data_bg.yaml" if with_bg else "data_base.yaml")
    y.write_text(f"path: {dst_root}\ntrain: {tr}\nval: images/val\ntest: images/test\nnames:\n{names}\n")
    return dst_root, y

def read_gt(label_path):
    a = np.loadtxt(label_path, ndmin=2) if Path(label_path).stat().st_size else np.zeros((0, 5))
    c = a[:, 0].astype(int); xc, yc, w, h = a[:, 1:5].T
    return c, np.stack([xc-w/2, yc-h/2, xc+w/2, yc+h/2], 1)

def predict_dets(model, img_paths, imgsz, conf=0.01, iou=NMS_IOU, batch=32, device=None, max_det=1000):
    """Detecciones de baja confianza; el barrido de umbral se hace despues sobre este cache."""
    out = {}
    paths = [str(p) for p in img_paths]
    for i in range(0, len(paths), batch):
        for p, r in zip(paths[i:i+batch], model.predict(paths[i:i+batch], imgsz=imgsz, conf=conf, iou=iou, verbose=False, device=device, max_det=max_det)):
            h, w = r.orig_shape
            b = r.boxes
            out[Path(p).stem] = (b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy(), b.xyxy.cpu().numpy() / np.array([w, h, w, h]))
    return out

def box_iou(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    i = iw * ih; ua = ((a[:, 2]-a[:, 0])*(a[:, 3]-a[:, 1]))[:, None] + ((b[:, 2]-b[:, 0])*(b[:, 3]-b[:, 1]))[None] - i
    return i / np.maximum(ua, 1e-9)

def load_gts(ds_root, split):
    return {p.stem: read_gt(p) for p in sorted((Path(ds_root) / "labels" / split).glob("*.txt"))}

def counts(dets, gts, conf, classes):
    """Conteos por imagen (gt, pred) de las clases dadas."""
    rows = []
    for k, (gc, _) in gts.items():
        c, s, _b = dets[k]
        rows.append((int(np.isin(gc, classes).sum()), int((np.isin(c, classes) & (s >= conf)).sum())))
    return np.array(rows)

def count_metrics(dets, gts, conf, classes):
    a = counts(dets, gts, conf, classes); e = a[:, 1] - a[:, 0]
    return dict(mae=float(np.abs(e).mean()), bias=float(e.mean()), exact=float((e == 0).mean()),
                gt_total=int(a[:, 0].sum()), pred_total=int(a[:, 1].sum()))

def boot_ci(dets, gts, conf, classes, n=1000, seed=0):
    a = counts(dets, gts, conf, classes); e = np.abs(a[:, 1] - a[:, 0]); rng = np.random.default_rng(seed)
    m = [e[rng.integers(0, len(e), len(e))].mean() for _ in range(n)]
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]

def boot_ci_groups(dets, gts, conf, classes, group_of, n=1000, seed=0, stat="mae"):
    """IC 95 % remuestreando GRUPOS de origen (group_of: stem -> grupo), no imagenes."""
    keys = list(gts); a = counts(dets, gts, conf, classes); e = np.abs(a[:, 1] - a[:, 0])
    g = np.array([group_of[k] for k in keys]); ug, inv = np.unique(g, return_inverse=True)
    sums = np.bincount(inv, weights=e, minlength=len(ug)); cnt = np.bincount(inv, minlength=len(ug)); rng = np.random.default_rng(seed)
    m = []
    for _ in range(n):
        ix = rng.integers(0, len(ug), len(ug)); m.append(sums[ix].sum() / cnt[ix].sum())
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]

def match_stats(dets, gts, conf, cls=GB, thr=0.5):
    """TP/FP/FN de la clase cls (IoU>=thr, 1 a 1 greedy) y confusiones con la otra clase de bacilos."""
    tp = fp = fn = conf_to_gpb = 0
    for k, (gc, gbx) in gts.items():
        c, s, b = dets[k]; m = s >= conf
        pc, pb = c[m], b[m]
        g = gc == cls; gb_ = gbx[g]; p = pb[pc == cls]
        iou = box_iou(p, gb_); used_p, used_g = set(), set()
        for v, i, j in sorted(((iou[i, j], i, j) for i in range(len(p)) for j in range(len(gb_))), reverse=True):
            if v < thr: break
            if i in used_p or j in used_g: continue
            used_p.add(i); used_g.add(j)
        t = len(used_p); tp += t; fp += len(p) - t; fn += len(gb_) - t
        other = GPB if cls == GB else GB
        un = [j for j in range(len(gb_)) if j not in used_g]
        if un:
            q = box_iou(pb[pc == other], gb_[un]); conf_to_gpb += int((q >= thr).any(0).sum()) if q.size else 0
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return dict(tp=tp, fp=fp, fn=fn, precision=P, recall=R, f1=2*P*R/max(P+R, 1e-9), gt_clase_predicha_como_otra=conf_to_gpb)

def sweep(dets, gts, classes, confs=np.round(np.arange(0.05, 0.81, 0.05), 2)):
    rows = []
    for c in confs:
        r = dict(conf=float(c), **count_metrics(dets, gts, c, classes))
        r.update({"f1_Gb": match_stats(dets, gts, c)["f1"]}); rows.append(r)
    return pd.DataFrame(rows)

def bg_fp(dets_bg, conf, classes=(GB,)):
    n = np.array([int((np.isin(c, classes) & (s >= conf)).sum()) for c, s, _ in dets_bg.values()])
    return dict(n_imgs=len(n), fp_por_imagen=float(n.mean()), imgs_con_fp=float((n > 0).mean()))

def cpu_latency_ms(model, img, imgsz, n=100, warm=10):
    for _ in range(warm): model.predict(img, imgsz=imgsz, device="cpu", verbose=False)
    t = []
    for _ in range(n):
        t0 = time.perf_counter(); model.predict(img, imgsz=imgsz, device="cpu", verbose=False); t.append((time.perf_counter()-t0)*1000)
    return float(np.median(t))

def ap_table(m):
    """mAP50 y mAP50-95 por clase desde un objeto DetMetrics de ultralytics."""
    idx = list(m.box.ap_class_index)
    return {CLASS_NAMES[c]: dict(ap50=float(m.box.ap50[i]), ap=float(m.box.ap[i])) for i, c in enumerate(idx)}

#!/usr/bin/env python
"""Referencia para la app: corre el .tflite de MicaTB (G-bacilli) con el mismo preprocesado,
decodificacion y NMS que debe hacer la app. Dependencias: numpy, opencv-python y ai-edge-litert
(o tflite-runtime / tensorflow). No usa ultralytics.

Uso: python inferir_tflite.py --model ecoli_gb_E3s43_fp16.tflite --config app_config.json --image campo.jpg [--definicion estricta|amplia]
Salida (JSON): conteo y cajas [x1,y1,x2,y2] en pixeles de la imagen original, con clase y confianza.
"""
import argparse, json
import cv2
import numpy as np


def cargar_interprete(path):
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        try:
            from tflite_runtime.interpreter import Interpreter
        except ImportError:
            from tensorflow.lite import Interpreter
    it = Interpreter(model_path=path); it.allocate_tensors()
    return it


def letterbox(bgr, size, fill=114):
    """Escala al lado mayor = size y rellena con gris centrado (igual que ultralytics, auto=False)."""
    h, w = bgr.shape[:2]
    r = min(size / h, size / w)
    nw, nh = round(w * r), round(h * r)
    dw, dh = (size - nw) / 2, (size - nh) / 2
    if (w, h) != (nw, nh):
        bgr = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top, bottom = round(dh - 0.1), round(dh + 0.1)
    left, right = round(dw - 0.1), round(dw + 0.1)
    out = cv2.copyMakeBorder(bgr, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(fill, fill, fill))
    return out, r, left, top


def preprocesar(bgr, size):
    img, r, left, top = letterbox(bgr, size)
    x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0   # RGB, 0..1
    return np.transpose(x, (2, 0, 1))[None], r, left, top               # NCHW


def nms_por_clase(boxes, scores, classes, iou_thr):
    """NMS greedy independiente por clase (suprime si IoU > umbral), como torchvision.batched_nms."""
    keep = []
    for c in np.unique(classes):
        idx = np.where(classes == c)[0]; idx = idx[np.argsort(-scores[idx])]
        while len(idx):
            i = idx[0]; keep.append(i)
            if len(idx) == 1: break
            rest = idx[1:]
            xx1 = np.maximum(boxes[i, 0], boxes[rest, 0]); yy1 = np.maximum(boxes[i, 1], boxes[rest, 1])
            xx2 = np.minimum(boxes[i, 2], boxes[rest, 2]); yy2 = np.minimum(boxes[i, 3], boxes[rest, 3])
            inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
            a1 = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
            a2 = (boxes[rest, 2] - boxes[rest, 0]) * (boxes[rest, 3] - boxes[rest, 1])
            idx = rest[inter / (a1 + a2 - inter + 1e-9) <= iou_thr]
    return np.array(keep, dtype=int)


def detectar(it, bgr, cfg, definicion="estricta"):
    size = cfg["entrada"]["tamano"]; H, W = bgr.shape[:2]
    x, r, left, top = preprocesar(bgr, size)
    i0, o0 = it.get_input_details()[0], it.get_output_details()[0]
    it.set_tensor(i0["index"], x); it.invoke()
    y = it.get_tensor(o0["index"])[0]                                   # [4 + n_clases, n_anclas]
    cx, cy, w, h = y[0] * size, y[1] * size, y[2] * size, y[3] * size   # cajas normalizadas 0..1 -> px de entrada
    boxes = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)
    scores_all = y[4:].T; cls = scores_all.argmax(1); conf = scores_all.max(1)
    d = cfg["definiciones"][definicion]
    m = conf > d["conf"]; boxes, conf, cls = boxes[m], conf[m], cls[m]
    k = nms_por_clase(boxes, conf, cls, d["iou_nms"])[: cfg["max_det"]]
    boxes, conf, cls = boxes[k], conf[k], cls[k]
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - left) / r; boxes[:, [1, 3]] = (boxes[:, [1, 3]] - top) / r   # deshacer letterbox
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, W); boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, H)
    clases_contadas = d["clases_contadas"]
    contar = np.isin(cls, clases_contadas)
    return dict(conteo=int(contar.sum()),
                detecciones=[dict(clase=cfg["clases"][int(c)], conf=round(float(s), 4), caja=[round(float(v), 1) for v in b])
                             for b, s, c in zip(boxes, conf, cls)])




# Verificaciones del modelo final (CPU): paridad, vecinos de id, respaldo 416, script de referencia y casos de prueba
Modelo: E3s43 (320 px). Umbrales congelados en val: estricta conf 0,60 / NMS 0,5; amplia conf 0,45 / NMS 0,5; max_det 1000.
No se usa test para elegir nada: solo para comprobar paridad, fuga y comportamiento.

## Entorno y datos

In [ ]:
import os, json, shutil, subprocess, sys, time, hashlib
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
subprocess.run("pip install -q ultralytics", shell=True)
from ultralytics import YOLO
from ultralytics.utils.metrics import compute_ap
OUT = Path("/kaggle/working/outputs"); OUT.mkdir(parents=True, exist_ok=True)
SRC = next(Path("/kaggle/input").glob("**/yolo_ds")); DSR = Path("/kaggle/temp/yolo_ds"); shutil.copytree(SRC, DSR)
find = lambda n: next(Path("/kaggle/input").glob(f"**/{n}"))
W3, W2 = find("E3s43_best.pt"), find("E2_best.pt")
TF = {k: find(f"ecoli_gb_E3s43{s}.tflite") for k, s in (("fp32", "_fp32"), ("fp16", "_fp16"), ("int8", "_int8"))}
names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
(DSR / "data_base.yaml").write_text(f"path: {DSR}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n{names}\n")
sp = pd.read_csv(DSR / "splits.csv", dtype={"dhash": str}); sp["stem"] = sp.name.str.replace(".jpg", "", regex=False); INFO = sp.set_index("stem")
GROUP_OF = INFO.group.to_dict()
IM = {s: sorted((DSR / "images" / s).glob("*.jpg")) for s in ("val", "test")}; GT = {s: load_gts(DSR, s) for s in ("val", "test")}
rng = np.random.default_rng(42); SUB = [IM["test"][i] for i in sorted(rng.choice(len(IM["test"]), 300, replace=False))]
G300 = {p.stem: GT["test"][p.stem] for p in SUB}
(DSR / "par300.txt").write_text("\n".join(map(str, SUB))); (DSR / "par300.yaml").write_text(f"path: {DSR}\ntrain: {DSR/'par300.txt'}\nval: {DSR/'par300.txt'}\ntest: {DSR/'par300.txt'}\nnames:\n{names}\n")
CONF, IOU = 0.6, 0.5
print("modelos:", W3, W2, TF)

## Funciones: AP50 propio sobre el camino de inferencia (predict) y metricas por imagen

In [ ]:
def ap50(dets, gts, cls=GB):
    rows, ngt = [], 0
    for k, (gc, gb) in gts.items():
        c, s, b = dets[k]; p = np.where(c == cls)[0]; g = np.where(gc == cls)[0]; ngt += len(g)
        order = p[np.argsort(-s[p])]; iou = box_iou(b[order], gb[g]); used = set()
        for i, pi in enumerate(order):
            best, bv = -1, 0.5
            for j in range(len(g)):
                if j not in used and iou[i, j] >= bv: best, bv = j, iou[i, j]
            if best >= 0: used.add(best)
            rows.append((s[pi], int(best >= 0)))
    rows.sort(key=lambda r: -r[0]); tp = np.cumsum([r[1] for r in rows]); fp = np.cumsum([1 - r[1] for r in rows])
    return float(compute_ap(tp / max(ngt, 1), tp / np.maximum(tp + fp, 1))[0])

def match_boxes(dets, gts, conf, cls=GB, thr=0.5):
    rows = []
    for k, (gc, gbx) in gts.items():
        c, s, b = dets[k]; m = s >= conf; pc, pb = c[m], b[m]
        g = np.where(gc == cls)[0]; p = np.where(pc == cls)[0]; iou = box_iou(pb[p], gbx[g]); up, ug = set(), set()
        for v, i, j in sorted(((iou[i, j], i, j) for i in range(len(p)) for j in range(len(g))), reverse=True):
            if v < thr: break
            if i in up or j in ug: continue
            up.add(i); ug.add(j)
        rows.append((k, len(up), len(p) - len(up), len(g) - len(ug), len(g), len(p)))
    return pd.DataFrame(rows, columns=["stem", "tp", "fp", "fn", "n_gt", "n_pred"]).set_index("stem")

def summ(mb):
    tp, fp, fn = mb.tp.sum(), mb.fp.sum(), mb.fn.sum(); P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return dict(n_img=len(mb), n_gt=int(mb.n_gt.sum()), precision=float(P), recall=float(R), f1=float(2 * P * R / max(P + R, 1e-9)), mae=float((mb.n_pred - mb.n_gt).abs().mean()))

## Bloque 1a. Paridad: ¿la diferencia 0,955 (test completo) vs 0,906 (300 imagenes, exportados) es muestreo, modo de validacion o conversion?
Se mide el mismo .pt en las mismas 300 imagenes con distintos modos de `val`, y con un AP50 propio sobre el camino de inferencia (predict), igual para pt y los tres .tflite.

In [ ]:
m3 = YOLO(str(W3)); par = {"val_ultralytics_pt_300": {}, "val_ultralytics_pt_test_completo": {}, "ap50_propio_predict_300": {}, "operacion_conf06_300": {}}
for rect in (True, False):
    for bs in (16, 1):
        v = m3.val(data=str(DSR / "par300.yaml"), split="val", imgsz=320, conf=0.001, iou=0.7, max_det=1000, device="cpu", batch=bs, rect=rect, verbose=False, plots=False)
        par["val_ultralytics_pt_300"][f"rect={rect},batch={bs}"] = float(v.box.ap50[list(v.box.ap_class_index).index(GB)])
for rect in (True, False):
    v = m3.val(data=str(DSR / "data_base.yaml"), split="test", imgsz=320, conf=0.001, iou=0.7, max_det=1000, device="cpu", batch=16, rect=rect, verbose=False, plots=False)
    par["val_ultralytics_pt_test_completo"][f"rect={rect}"] = float(v.box.ap50[list(v.box.ap_class_index).index(GB)])
models = {"pt": m3, **{f"tflite_{k}": YOLO(str(p), task="detect") for k, p in TF.items()}}
DET300 = {}
for k, mod in models.items():
    DET300[k] = predict_dets(mod, SUB, 320, conf=0.01, iou=IOU, device="cpu", batch=1)
    par["ap50_propio_predict_300"][k] = ap50(DET300[k], G300); par["operacion_conf06_300"][k] = summ(match_boxes(DET300[k], G300, CONF))
DET_T = predict_dets(m3, IM["test"], 320, conf=0.01, iou=IOU, device="cpu", batch=1); par["ap50_propio_predict_test_completo_pt"] = ap50(DET_T, GT["test"])
(OUT / "paridad_diagnostico.json").write_text(json.dumps(par, indent=1)); print(json.dumps(par, indent=1))

## Bloque 1b. Fuga por ids vecinos: rendimiento en test segun si hay un vecino de id (distancia 1 o 2) en train

In [ ]:
tr_ids = set(INFO[INFO.split == "train"].group_id.astype(int)); mb_t = match_boxes(DET_T, GT["test"], CONF)
ids = INFO.loc[mb_t.index].group_id.astype(int)
near = lambda i, d: any((i + o) in tr_ids for o in range(-d, d + 1) if o != 0)
mb_t["vec1"] = [near(i, 1) for i in ids]; mb_t["vec2"] = [near(i, 2) for i in ids]; mb_t["grupo"] = mb_t.index.map(GROUP_OF)
res = {}
def boot_diff(flag, n=1000, seed=0):
    a = mb_t.assign(f=mb_t[flag]).groupby(["grupo", "f"])[["tp", "fp", "fn", "n_gt", "n_pred"]].sum().reset_index(); gs = a.grupo.unique(); rng = np.random.default_rng(seed); d = []
    by = {g: a[a.grupo == g] for g in gs}
    for _ in range(n):
        s = pd.concat([by[g] for g in rng.choice(gs, len(gs))]); r = {}
        for f in (True, False):
            t = s[s.f == f][["tp", "fp", "fn"]].sum(); P = t.tp / max(t.tp + t.fp, 1); R = t.tp / max(t.tp + t.fn, 1); r[f] = (R, 2 * P * R / max(P + R, 1e-9))
        d.append((r[True][0] - r[False][0], r[True][1] - r[False][1]))
    q = np.percentile(np.array(d), [2.5, 97.5], axis=0); return dict(recall_dif_ic95=[float(q[0, 0]), float(q[1, 0])], f1_dif_ic95=[float(q[0, 1]), float(q[1, 1])])
for flag in ("vec1", "vec2"):
    res[flag] = {"con_vecino": summ(mb_t[mb_t[flag]]), "sin_vecino": summ(mb_t[~mb_t[flag]]), **boot_diff(flag)}
res["por_patron"] = {p: {"n": int((INFO.loc[mb_t.index].pattern == p).sum()), "con_vecino_dist2": int(mb_t[INFO.loc[mb_t.index].pattern == p].vec2.sum())} for p in ("crop", "simple")}
(OUT / "vecinos_id_test.json").write_text(json.dumps(res, indent=1)); print(json.dumps(res, indent=1))

## Bloque 1c. Respaldo 416 px (E2): umbrales en val, exportacion ONNX/FP32/FP16 y paridad en las 300 imagenes

In [ ]:
m2 = YOLO(str(W2)); dv = {i: predict_dets(m2, IM["val"], 416, conf=0.01, iou=i, device="cpu") for i in (0.5, 0.6, 0.7)}
g = pd.DataFrame([dict(iou=i, conf=float(c), mae=count_metrics(dv[i], GT["val"], c, [GB])["mae"]) for i in dv for c in np.round(np.arange(0.05, 0.701, 0.05), 2)])
bst = g.sort_values(["mae", "iou", "conf"]).iloc[0]; thr416 = dict(conf=float(bst.conf), iou_nms=float(bst.iou), mae_val=float(bst.mae))
EX = Path("/kaggle/working/export416"); EX.mkdir(exist_ok=True); w = EX / "ecoli_gb_E2_416.pt"; shutil.copy2(W2, w)
def exp(tag, kw):
    r = subprocess.run([sys.executable, "-c", f"from ultralytics import YOLO; print(YOLO('{w}').export(imgsz=416, {kw}))"], capture_output=True, text=True, timeout=1500, cwd=str(EX)); print(tag, r.returncode, (r.stdout + r.stderr)[-200:])
exp("onnx", "format='onnx', simplify=True"); exp("tflite fp32", "format='tflite'")
shutil.copy2(EX / "ecoli_gb_E2_416.tflite", EX / "ecoli_gb_E2_416_fp32.tflite")
from ai_edge_quantizer import quantizer, qtyping
q = quantizer.Quantizer(str(EX / "ecoli_gb_E2_416_fp32.tflite"))
q.update_quantization_recipe(regex=".*", operation_name=qtyping.TFLOperationName.ALL_SUPPORTED, algorithm_key="float_casting",
    op_config=qtyping.OpQuantizationConfig(weight_tensor_config=qtyping.TensorQuantizationConfig(num_bits=16, dtype=qtyping.TensorDataType.FLOAT), compute_precision=qtyping.ComputePrecision.FLOAT, explicit_dequantize=True))
q.quantize().export_model(str(EX / "ecoli_gb_E2_416_fp16.tflite"), overwrite=True)
r416 = {"umbrales_val": thr416, "paridad_300": {}}
for k, p in (("pt", w), ("tflite_fp16", EX / "ecoli_gb_E2_416_fp16.tflite")):
    d = predict_dets(YOLO(str(p), task="detect"), SUB, 416, conf=0.01, iou=thr416["iou_nms"], device="cpu", batch=1)
    r416["paridad_300"][k] = dict(ap50_propio=ap50(d, G300), **summ(match_boxes(d, G300, thr416["conf"])))
(OUT / "respaldo416.json").write_text(json.dumps(r416, indent=1)); print(json.dumps(r416, indent=1))
for f in EX.glob("ecoli_gb_E2_416*"):
    if f.suffix in (".pt", ".onnx", ".tflite"): shutil.copy2(f, OUT / f.name)

## Bloque 2. Config de la app, script de referencia y verificacion contra ultralytics (300 imagenes de test)

In [ ]:
APP = {"modelo": "ecoli_gb_E3s43_fp16.tflite", "version": "1.0", "clases": CLASS_NAMES, "max_det": 1000,
       "entrada": {"tamano": 320, "forma": [1, 3, 320, 320], "orden": "NCHW", "canales": "RGB", "dtype": "float32", "normalizacion": "pixel/255, sin media ni desviacion", "letterbox": "lado mayor=320, relleno gris 114 centrado, interpolacion bilineal"},
       "salida": {"forma": [1, 8, 2100], "filas_0_3": "cx,cy,w,h normalizados 0-1 respecto a 320", "filas_4_7": "score por clase 0-1 (sin sigmoide)", "nms_incluido": False},
       "definiciones": {"estricta": {"clases_contadas": [2], "conf": 0.60, "iou_nms": 0.5}, "amplia": {"clases_contadas": [2, 3], "conf": 0.45, "iou_nms": 0.5}},
       "definicion_recomendada": "estricta", "nms": "por clase; se filtra con max(score de clases) > conf; se cuenta solo si la clase ganadora esta en clases_contadas"}
(OUT / "app_config.json").write_text(json.dumps(APP, indent=1, ensure_ascii=False))
it = cargar_interprete(str(TF["fp16"])); R = []
for p in SUB:
    img = cv2.imread(str(p))
    mine = detectar(it, img, APP, "estricta"); R.append((p.stem, mine))
uc = YOLO(str(TF["fp16"]), task="detect"); agree, maxdiff, diffs = 0, 0.0, []
for (st, mine), p in zip(R, SUB):
    r = uc.predict(str(p), imgsz=320, conf=CONF, iou=IOU, device="cpu", verbose=False, max_det=1000)[0]
    n_u = int((r.boxes.cls.cpu().numpy() == GB).sum()); agree += int(n_u == mine["conteo"])
    bu = r.boxes.xyxy.cpu().numpy()[r.boxes.cls.cpu().numpy() == GB]; bm = np.array([d["caja"] for d in mine["detecciones"] if d["clase"] == "G-bacilli"]).reshape(-1, 4)
    if len(bu) == len(bm) and len(bu): diffs.append(float(np.abs(np.sort(bu, 0) - np.sort(bm, 0)).max()))
ver = dict(imagenes=len(R), conteo_igual_a_ultralytics=agree, max_dif_caja_px=float(max(diffs)) if diffs else None, mediana_dif_caja_px=float(np.median(diffs)) if diffs else None)
print("script vs ultralytics:", ver)

## Casos de prueba (20 imagenes de test con resultado esperado) y 3 recortes no cuadrados (prueba del letterbox)

In [ ]:
CAS = OUT / "casos"; CAS.mkdir(exist_ok=True); rs = np.random.default_rng(7); stems = [p.stem for p in IM["test"]]; ngt = {s: int((GT["test"][s][0] == GB).sum()) for s in stems}
pick = []
for lo, hi, n in ((0, 0, 4), (1, 1, 6), (2, 3, 5), (4, 99, 5)):
    pool = [s for s in stems if lo <= ngt[s] <= hi]; pick += list(rs.choice(pool, n, replace=False))
casos = []
import io
from IPython.display import display, Image as IPImage
def panel(img_bgr, gc, gb, dets):
    gt_v = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy(); pr_v = gt_v.copy()
    for c, b in zip(gc, gb):
        x1, y1, x2, y2 = (b * 640).astype(int); cv2.rectangle(gt_v, (x1, y1), (x2, y2), (0, 200, 0) if c == GB else (255, 160, 0), 2)
    for d in dets:
        x1, y1, x2, y2 = [int(v) for v in d["caja"]]; col = (255, 0, 0) if d["clase"] == "G-bacilli" else (150, 0, 255)
        cv2.rectangle(pr_v, (x1, y1), (x2, y2), col, 2); cv2.putText(pr_v, f'{d["conf"]:.2f}', (x1, max(y1 - 6, 14)), 0, 0.6, col, 2)
    return gt_v, pr_v
print("Leyenda: verde = G-bacilli real (etiqueta); naranja = otra clase real; rojo = prediccion G-bacilli (con confianza); violeta = prediccion de otra clase")
for n_, s in enumerate(pick, 1):
    img = cv2.imread(str(DSR / "images" / "test" / f"{s}.jpg")); cv2.imwrite(str(CAS / f"{s}.jpg"), img)
    res_ = detectar(it, img, APP, "estricta"); res_a = detectar(it, img, APP, "amplia"); gc, gb = GT["test"][s]
    gtb = [[round(float(v), 1) for v in (b * 640)] for c, b in zip(gc, gb) if c == GB]
    dg = [d for d in res_["detecciones"] if d["clase"] == "G-bacilli"]
    iou_ = box_iou(np.array([d["caja"] for d in dg]).reshape(-1, 4) / 640, np.array(gtb).reshape(-1, 4) / 640); tp_ = len({j for i, j in zip(*np.where(iou_ >= 0.5))}) if iou_.size else 0
    casos.append(dict(imagen=f"{s}.jpg", gt_G_bacilli=len(gtb), gt_cajas_G_bacilli=gtb, esperado_script_estricta=res_["conteo"], esperado_script_amplia=res_a["conteo"], detecciones_estricta=dg, patron=INFO.pattern[s]))
    print(f"[{n_:02d}/20] {s}.jpg ({INFO.pattern[s]}) | etiqueta real G-bacilli = {len(gtb)} | prediccion del modelo = {res_['conteo']} (confianzas {[d['conf'] for d in dg]}) | aciertos~{tp_} falsos+~{len(dg) - tp_} perdidos~{len(gtb) - tp_}")
    gt_v, pr_v = panel(img, gc, gb, res_["detecciones"])
    fig, ax = plt.subplots(1, 2, figsize=(12, 6.2)); ax[0].imshow(gt_v); ax[0].set_title(f"ETIQUETA REAL: {len(gtb)} G-bacilli", fontsize=13); ax[1].imshow(pr_v); ax[1].set_title(f"PREDICCION DEL MODELO: {res_['conteo']} G-bacilli", fontsize=13)
    for x in ax: x.axis("off")
    plt.tight_layout(); buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=70); plt.close(fig); display(IPImage(data=buf.getvalue()))
    if n_ <= 20: open(CAS / f"panel_{n_:02d}_{s}.png", "wb").write(buf.getvalue())
nosq = []
for k, s in enumerate(pick[4:7]):
    img = cv2.imread(str(DSR / "images" / "test" / f"{s}.jpg"))[:, :, :]; crop = img[80:560, 0:640] if k == 0 else (img[0:640, 100:500] if k == 1 else img[160:400, 20:620])
    cv2.imwrite(str(CAS / f"nocuadrada_{k}_{s}.jpg"), crop); mine = detectar(it, crop, APP, "estricta")
    r = uc.predict(str(CAS / f"nocuadrada_{k}_{s}.jpg"), imgsz=320, conf=CONF, iou=IOU, device="cpu", verbose=False, max_det=1000)[0]
    bu = r.boxes.xyxy.cpu().numpy()[r.boxes.cls.cpu().numpy() == GB]; bm = np.array([d["caja"] for d in mine["detecciones"] if d["clase"] == "G-bacilli"]).reshape(-1, 4)
    nosq.append(dict(imagen=f"nocuadrada_{k}_{s}.jpg", forma=list(crop.shape[:2]), conteo_script=mine["conteo"], conteo_ultralytics=int(len(bu)),
                     max_dif_caja_px=float(np.abs(np.sort(bu, 0) - np.sort(bm, 0)).max()) if len(bu) == len(bm) and len(bu) else None, detecciones=mine["detecciones"]))
(OUT / "casos_prueba.json").write_text(json.dumps(dict(config="app_config.json", modelo="ecoli_gb_E3s43_fp16.tflite", verificacion_vs_ultralytics_300=ver, casos=casos, no_cuadradas=nosq), indent=1, ensure_ascii=False))
tab = pd.DataFrame([(c["imagen"], c["patron"], c["gt_G_bacilli"], c["esperado_script_estricta"], c["esperado_script_amplia"]) for c in casos], columns=["imagen", "patron", "GT", "modelo_estricta", "modelo_amplia"])
tab["error"] = tab.modelo_estricta - tab.GT; print(tab.to_string()); print("MAE en estos 20:", tab.error.abs().mean(), "| no cuadradas:", json.dumps([{k: v for k, v in n.items() if k != "detecciones"} for n in nosq]))
tab.to_csv(OUT / "casos_prueba_tabla.csv", index=False)
open(OUT / "inferir_tflite.py", "w", encoding="utf8").write(INFERIR_SRC)